<img src="https://raw.githubusercontent.com/unslothai/unsloth/main/images/unsloth%20new%20logo.png" width="200"/>

# Unsloth × ExLlamaV3 (EXL3)
## Qwen3.6-35B-A3B (MoE) at 2-bit on a single 24 GB GPU

EXL3 is Unsloth's quantization backend, replacing bitsandbytes. Unlike
bitsandbytes (4/8-bit only, and **no MoE** under transformers 5), EXL3
supports 2/3/4/6/8-bit and quantizes Mixture-of-Experts models.

Here we take **`Qwen/Qwen3.6-35B-A3B`** - a 35B Mixture-of-Experts model
- quantize it to **2-bit**, and QLoRA-fine-tune it entirely on one 24 GB
GPU (e.g. an RTX 3090/4090).

| | Value |
|---|---|
| Model | Qwen/Qwen3.6-35B-A3B (MoE) |
| Quantization | EXL3 2-bit decoder / 4-bit head |
| GPU | one 24 GB card |

> A big MoE's experts would OOM if reconstructed to dense tensors.
> Unsloth auto-detects large MoEs (>32 experts/layer) and keeps each
> expert EXL3-quantized, reconstructing only the top-k experts a batch
> routes to, on the fly - so the 35B fits in VRAM.

In [ ]:
# Install Unsloth with the EXL3 (ExLlamaV3) backend. Requires a CUDA GPU
# and a CUDA 12.4+ build of PyTorch.
%pip install -q "unsloth[exllama]"

### 1. Load & quantize to 2-bit

The first run quantizes and caches the checkpoint; later runs load it
instantly. `calibrate=True` gives the best 2-bit quality.

In [ ]:
import torch
from unsloth import FastLanguageModel
from unsloth.exllama import Exl3Config

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3.6-35B-A3B",
    max_seq_length = 2048,
    dtype = torch.bfloat16,   # newest Qwen MoEs need bfloat16 compute
    load_in_exl3 = Exl3Config(bits = 2.0, head_bits = 4, calibrate = True),
)

### 2. Quick generation sanity check

In [ ]:
FastLanguageModel.for_inference(model)
messages = [{"role": "user", "content": "Write one vivid sentence about the sea."}]
ids = tokenizer.apply_chat_template(
    messages, add_generation_prompt = True, return_tensors = "pt",
).to("cuda")
out = model.generate(input_ids = ids, max_new_tokens = 64, do_sample = False)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens = True))

### 3. Attach LoRA adapters

Target the attention + shared-MLP projections; the routed experts stay
frozen-quantized (standard QLoRA-on-MoE).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r = 32, lora_alpha = 64, lora_dropout = 0.0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth", random_state = 3407,
)
model.print_trainable_parameters()

### Fine-tune with LoRA

A tiny slice of a dataset keeps this quick; swap in your own for real runs.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt

dataset = load_dataset("mlabonne/FineTome-100k", split = "train[:500]")
dataset = standardize_sharegpt(dataset)

def fmt(ex):
    return {
        "text": tokenizer.apply_chat_template(
            ex["conversations"], tokenize = False, add_generation_prompt = False,
        ),
    }
dataset = dataset.map(fmt)

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model, train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 8, max_steps = 60, learning_rate = 3e-4,
        logging_steps = 5, optim = "adamw_torch", lr_scheduler_type = "cosine",
        seed = 3407, output_dir = "outputs", report_to = "none",
    ),
)
trainer.train()

### 4. Save the adapter

The small LoRA adapter reloads onto the 2-bit base for inference or
further training.

In [ ]:
model.save_pretrained("Qwen3.6-35B-A3B-exl3-2bit-lora")
tokenizer.save_pretrained("Qwen3.6-35B-A3B-exl3-2bit-lora")
print("saved LoRA adapter")